In [2]:
import csv
import pandas as pd
import pingouin as pg
import numpy as np
import json
from scipy import stats

#### Config

In [3]:
Data_Root = '/home/bingkzhao2/Data/Patents/'
ALPHA_THRESHOLD = 0.59

#### Load Data

In [4]:
df = pd.read_csv(Data_Root + 'patent_reg_data.csv', header=0, low_memory=False)
print(f"Data loaded: {df.shape[0]} observations")

Data loaded: 2748927 observations


In [5]:
hype_words_li = ['broad', 'important', 'surprising', 'dire', 'latest', 'fastest', 'exciting', 'devastating', 'extensive', 'revolutionary', 'enormous', 'unparalleled', 'reproducible', 'unique', 'premier', 'durable', 'effective', 'tremendous', 'efficient', 'expansive', 'novel', 'sophisticated', 'alarming', 'interdisciplinary', 'ready', 'skilled', 'synergistic', 'greatest', 'stark', 'significant', 'pivotal', 'emerging', 'dedicated', 'relevant', 'desperate', 'qualified', 'easy', 'attractive', 'incredible', 'meaningful', 'exceptional', 'timely', 'advanced', 'accurate', 'nuanced', 'motivated', 'critical', 'substantial', 'outstanding', 'systematic', 'scalable', 'daunting', 'promising', 'comprehensive', 'unprecedented', 'biggest', 'intuitive', 'compelling', 'scientific', 'considerable', 'international', 'massive', 'fundamental', 'intriguing', 'major', 'strategic', 'dismal', 'urgent', 'huge', 'transdisciplinary', 'vibrant', 'efficacious', 'immediate', 'multidisciplinary', 'user-friendly', 'diverse', 'vast', 'innovative', 'robust', 'elusive', 'transformative', 'ultimate', 'powerful', 'ambitious', 'detailed', 'seamless', 'actionable', 'confident', 'interesting', 'safer', 'successful', 'careful', 'immense', 'crucial', 'groundbreaking', 'essential', 'notable', 'myriad', 'rigorous', 'generalizable', 'collegial', 'interprofessional', 'tailored', 'talented', 'longstanding', 'largest', 'strong', 'top', 'intellectual', 'remarkable', 'unmet', 'deeper', 'unanswered', 'deployable', 'rich', 'experienced', 'paramount', 'renowned', 'impactful', 'creative', 'cohesive', 'indispensable', 'ideal', 'productive', 'overwhelming', 'senior', 'tangible', 'sustainable', 'foundational', 'invaluable', 'key', 'prestigious', 'vital', 'accessible', 'stellar', 'imperative', 'first', 'ample', 'quality']

#### Cronbach's Alpha Iterative Screening

In [6]:
valid_hype_words = [word for word in hype_words_li if word in df.columns]
hype_df = df[valid_hype_words].copy()
hype_df.dropna(inplace=True)
print(f"Valid samples for analysis: {hype_df.shape[0]}")

# Begin iterative Screening
print(f"\nCronbach's Alpha Screening (Target: >= {ALPHA_THRESHOLD})")
print(f"Initial words: {len(valid_hype_words)}")

words_to_keep = valid_hype_words.copy()
initial_alpha = pg.cronbach_alpha(data=hype_df[words_to_keep])[0]
current_alpha = initial_alpha
print(f"Initial Alpha: {current_alpha:.4f}")

if current_alpha >= ALPHA_THRESHOLD:
    print("Initial Alpha meets threshold. No screening needed.")
else:
    iteration = 0
    while current_alpha < ALPHA_THRESHOLD:
        iteration += 1
        alpha_if_dropped = {}
        
        # Calculate alpha if each word is dropped
        for word in words_to_keep:
            temp_list = [w for w in words_to_keep if w != word]
            alpha_val = pg.cronbach_alpha(data=hype_df[temp_list])[0]
            alpha_if_dropped[word] = alpha_val
            
        # Find word that gives highest alpha when dropped
        word_to_drop = max(alpha_if_dropped, key=alpha_if_dropped.get)
        max_alpha_after_drop = alpha_if_dropped[word_to_drop]
        
        # Only drop if it improves alpha
        if max_alpha_after_drop > current_alpha:
            words_to_keep.remove(word_to_drop)
            current_alpha = max_alpha_after_drop
            print(f"Iteration {iteration}: Dropped '{word_to_drop}' | Alpha: {current_alpha:.4f} | Words: {len(words_to_keep)}")
        else:
            print("No further improvement possible. Screening stopped.")
            break

Valid samples for analysis: 2748927

Cronbach's Alpha Screening (Target: >= 0.59)
Initial words: 139
Initial Alpha: 0.1824
Iteration 1: Dropped 'first' | Alpha: 0.4111 | Words: 138
Iteration 2: Dropped 'key' | Alpha: 0.5042 | Words: 137
Iteration 3: Dropped 'top' | Alpha: 0.5664 | Words: 136
Iteration 4: Dropped 'quality' | Alpha: 0.5948 | Words: 135


In [7]:
dropped_words = [word for word in valid_hype_words if word not in words_to_keep]  
print(f"\nDropped words: {dropped_words}")  


Dropped words: ['top', 'key', 'first', 'quality']


In [8]:
df_revised = df.drop(columns=dropped_words, errors="ignore")

In [9]:
df_revised['total_num_hype_words_revised'] = df_revised[words_to_keep].sum(axis=1)
df_revised['frac_hype_words_revised'] = (
    df_revised['total_num_hype_words_revised'] 
    / df_revised['num_words']
)

#### Print the results

In [10]:
output_csv = Data_Root + 'patent_reg_data_revised.csv'
df_revised.to_csv(output_csv, index=False)
print(f"\nFinal Results:")
print(f"Final Alpha: {current_alpha:.4f}")
print(f"Final word count: {len(words_to_keep)}")
print(f"Words retained: {len(words_to_keep)}/{len(valid_hype_words)}")
print(f"Saved CSV: {output_csv}")


Final Results:
Final Alpha: 0.5948
Final word count: 135
Words retained: 135/139
Saved CSV: /home/bingkzhao2/Data/Patents/patent_reg_data_revised.csv


In [12]:
column_names = df_revised.columns.tolist()
print(column_names)

['pgpub_id', 'year', 'application_id', 'cpc_group', 'cpc_class', 'cpc_subclass', 'num_cpc_code', 'novelty_min', 'novelty_median', 'applicant_entity', 'decision', 'patent_id', 'patent_year', 'num_attorneys', 'attorney_num_prior_apps', 'attorney_num_prior_patents', 'examiner_num_prior_apps', 'num_inventors', 'num_prior_apps', 'num_prior_patents', 'team_gender', 'attorney_gender', 'examiner_gender', 'description_length', 'flesch_reading_ease', 'ambitious', 'scientific', 'rigorous', 'broad', 'latest', 'deployable', 'detailed', 'transdisciplinary', 'devastating', 'experienced', 'elusive', 'strong', 'interdisciplinary', 'comprehensive', 'strategic', 'timely', 'fastest', 'deeper', 'alarming', 'unmet', 'transformative', 'stellar', 'accurate', 'vast', 'productive', 'generalizable', 'tailored', 'sophisticated', 'careful', 'dire', 'advanced', 'outstanding', 'user-friendly', 'imperative', 'huge', 'intuitive', 'surprising', 'premier', 'successful', 'unprecedented', 'senior', 'intriguing', 'immense'